# Reconciliation Anomaly Detection

## Objective
Build an anomaly detection system to identify unusual variances in financial reconciliations before they become problems. This notebook demonstrates:

1. **Exploratory Data Analysis (EDA)** - Understanding variance patterns
2. **Feature Engineering** - Creating a feature store view
3. **Model Development** - Multiple algorithms with hyperparameter tuning
4. **Experiment Tracking** - MLflow-style tracking with Snowflake
5. **Model Registry** - Version management and promotion
6. **Cross-Validation** - Proper temporal splits to prevent data leakage

---

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, when, lit, abs as sf_abs, avg, stddev, max as sf_max, min as sf_min, count, sum as sf_sum, lag, datediff
from snowflake.snowpark.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print(f"Session active: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

---
## 1. Exploratory Data Analysis (EDA)

Let's understand the reconciliation data and variance patterns.

In [ ]:
%%sql -r data_overview
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT assignment_id) as unique_assignments,
    COUNT(DISTINCT entity_name) as unique_entities,
    COUNT(DISTINCT period_end_date) as unique_periods,
    MIN(period_end_date) as earliest_period,
    MAX(period_end_date) as latest_period
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360

In [ ]:
%%sql -r status_dist
SELECT 
    reconciliation_status,
    COUNT(*) as count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as pct,
    AVG(total_abs_variance) as avg_variance,
    MAX(total_abs_variance) as max_variance
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY reconciliation_status
ORDER BY count DESC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
status_dist.plot(kind='bar', x='RECONCILIATION_STATUS', y='COUNT', ax=ax1, color='steelblue', legend=False)
ax1.set_title('Reconciliation Status Distribution', fontsize=12)
ax1.set_xlabel('Status')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

ax2 = axes[1]
status_dist.plot(kind='bar', x='RECONCILIATION_STATUS', y='AVG_VARIANCE', ax=ax2, color='coral', legend=False)
ax2.set_title('Average Variance by Status', fontsize=12)
ax2.set_xlabel('Status')
ax2.set_ylabel('Avg Variance ($)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r variance_trend
SELECT 
    period_end_date,
    period_year,
    period_quarter,
    COUNT(*) as total_records,
    SUM(CASE WHEN reconciliation_status = 'High Variance' THEN 1 ELSE 0 END) as high_variance_count,
    AVG(total_abs_variance) as avg_variance,
    STDDEV(total_abs_variance) as stddev_variance,
    MAX(total_abs_variance) as max_variance,
    PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_abs_variance) as p95_variance
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY period_end_date, period_year, period_quarter
ORDER BY period_end_date

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
ax1.plot(variance_trend['PERIOD_END_DATE'], variance_trend['AVG_VARIANCE'], marker='o', color='steelblue')
ax1.fill_between(variance_trend['PERIOD_END_DATE'], 
                  variance_trend['AVG_VARIANCE'] - variance_trend['STDDEV_VARIANCE'],
                  variance_trend['AVG_VARIANCE'] + variance_trend['STDDEV_VARIANCE'],
                  alpha=0.3)
ax1.set_title('Average Variance Over Time (with Std Dev)', fontsize=12)
ax1.set_xlabel('Period')
ax1.set_ylabel('Variance ($)')
ax1.tick_params(axis='x', rotation=45)

ax2 = axes[0, 1]
ax2.bar(variance_trend['PERIOD_END_DATE'], variance_trend['HIGH_VARIANCE_COUNT'], color='coral')
ax2.set_title('High Variance Count by Period', fontsize=12)
ax2.set_xlabel('Period')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

ax3 = axes[1, 0]
ax3.plot(variance_trend['PERIOD_END_DATE'], variance_trend['P95_VARIANCE'], marker='s', color='green')
ax3.set_title('95th Percentile Variance Trend', fontsize=12)
ax3.set_xlabel('Period')
ax3.set_ylabel('P95 Variance ($)')
ax3.tick_params(axis='x', rotation=45)

ax4 = axes[1, 1]
ax4.scatter(variance_trend['TOTAL_RECORDS'], variance_trend['AVG_VARIANCE'], 
            c=variance_trend['PERIOD_YEAR'], cmap='viridis', s=100)
ax4.set_title('Variance vs Record Count (colored by year)', fontsize=12)
ax4.set_xlabel('Total Records')
ax4.set_ylabel('Avg Variance ($)')

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r variance_buckets
SELECT 
    CASE 
        WHEN total_abs_variance = 0 THEN '0 - Zero'
        WHEN total_abs_variance < 100 THEN '1 - Under $100'
        WHEN total_abs_variance < 1000 THEN '2 - $100-$1K'
        WHEN total_abs_variance < 10000 THEN '3 - $1K-$10K'
        WHEN total_abs_variance < 100000 THEN '4 - $10K-$100K'
        ELSE '5 - Over $100K'
    END as variance_bucket,
    COUNT(*) as count,
    AVG(balance_gl) as avg_gl_balance,
    AVG(balance_bank) as avg_bank_balance
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY 1
ORDER BY 1

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
variance_buckets.plot(kind='bar', x='VARIANCE_BUCKET', y='COUNT', ax=ax, color='steelblue', logy=True)
ax.set_title('Variance Distribution (Log Scale)', fontsize=12)
ax.set_xlabel('Variance Bucket')
ax.set_ylabel('Count (Log Scale)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("\nVariance Distribution Summary:")
print(variance_buckets.to_string(index=False))

---
## 2. Feature Engineering & Feature Store

Create features that capture:
- **Variance characteristics**: magnitude, volatility, trends
- **Historical patterns**: rolling averages, period-over-period changes
- **Account characteristics**: balance ratios, activity levels
- **Entity context**: hierarchy level, key account status

We'll create a **Feature Store View** that can be reused for model training and inference.

In [ ]:
feature_store_sql = """
CREATE OR REPLACE VIEW COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES AS
WITH base_features AS (
    SELECT 
        r.assignment_id,
        r.period_id,
        r.period_end_date,
        r.entity_id,
        r.entity_name,
        r.account_combination,
        r.currency,
        r.reconciliation_status,
        r.is_active,
        r.is_key_account,
        r.hierarchy_depth,
        COALESCE(r.balance_gl, 0) as balance_gl,
        COALESCE(r.balance_bank, 0) as balance_bank,
        COALESCE(r.balance_subledger, 0) as balance_subledger,
        COALESCE(r.balance_estimate, 0) as balance_estimate,
        COALESCE(r.total_abs_variance, 0) as variance_amount,
        COALESCE(r.gl_bank_difference, 0) as gl_bank_diff,
        COALESCE(r.gl_subledger_difference, 0) as gl_subledger_diff,
        COALESCE(r.reconciliation_count, 0) as recon_count,
        COALESCE(r.total_unidentified_amount, 0) as unidentified_amount
    FROM COCO_LIVE_DB.DBT.RECONCILIATION_360 r
    WHERE r.is_active = TRUE
),
windowed_features AS (
    SELECT 
        b.*,
        LAG(b.variance_amount, 1) OVER (PARTITION BY b.assignment_id ORDER BY b.period_end_date) as prev_variance,
        LAG(b.variance_amount, 2) OVER (PARTITION BY b.assignment_id ORDER BY b.period_end_date) as prev_variance_2,
        LAG(b.balance_gl, 1) OVER (PARTITION BY b.assignment_id ORDER BY b.period_end_date) as prev_balance_gl,
        AVG(b.variance_amount) OVER (
            PARTITION BY b.assignment_id ORDER BY b.period_end_date ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) as rolling_avg_variance_3,
        STDDEV(b.variance_amount) OVER (
            PARTITION BY b.assignment_id ORDER BY b.period_end_date ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) as rolling_std_variance_3,
        MAX(b.variance_amount) OVER (
            PARTITION BY b.assignment_id ORDER BY b.period_end_date ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING
        ) as rolling_max_variance_6,
        AVG(b.variance_amount) OVER (PARTITION BY b.entity_id, b.period_end_date) as entity_avg_variance,
        COUNT(*) OVER (PARTITION BY b.entity_id, b.period_end_date) as entity_assignment_count,
        AVG(b.variance_amount) OVER (PARTITION BY b.period_end_date) as period_avg_variance,
        STDDEV(b.variance_amount) OVER (PARTITION BY b.period_end_date) as period_std_variance,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY b.variance_amount) OVER (PARTITION BY b.period_end_date) as period_p95_variance
    FROM base_features b
)
SELECT 
    w.*,
    CASE WHEN COALESCE(w.prev_variance, 0) > 0 THEN (w.variance_amount - w.prev_variance) / w.prev_variance * 100 ELSE 0 END as variance_pct_change,
    CASE WHEN COALESCE(w.rolling_std_variance_3, 0) > 0 THEN (w.variance_amount - COALESCE(w.rolling_avg_variance_3, 0)) / w.rolling_std_variance_3 ELSE 0 END as variance_z_score,
    CASE WHEN COALESCE(w.period_std_variance, 0) > 0 THEN (w.variance_amount - COALESCE(w.period_avg_variance, 0)) / w.period_std_variance ELSE 0 END as variance_z_score_period,
    CASE WHEN ABS(w.balance_gl) > 0 THEN w.variance_amount / ABS(w.balance_gl) * 100 ELSE 0 END as variance_pct_of_balance,
    CASE WHEN COALESCE(w.rolling_max_variance_6, 0) > 0 THEN w.variance_amount / w.rolling_max_variance_6 ELSE 0 END as variance_vs_rolling_max,
    CASE WHEN w.variance_amount > COALESCE(w.period_p95_variance, 0) THEN 1 ELSE 0 END as is_above_p95,
    CASE WHEN ABS(w.balance_gl) > 0 THEN w.gl_bank_diff / ABS(w.balance_gl) * 100 ELSE 0 END as gl_bank_diff_ratio,
    CASE WHEN COALESCE(w.prev_balance_gl, 0) != 0 THEN (w.balance_gl - w.prev_balance_gl) / ABS(w.prev_balance_gl) * 100 ELSE 0 END as balance_change_pct,
    CASE w.hierarchy_depth WHEN 1 THEN 0.2 WHEN 2 THEN 0.4 WHEN 3 THEN 0.6 WHEN 4 THEN 0.8 ELSE 1.0 END as hierarchy_depth_normalized,
    CASE WHEN w.is_key_account THEN 1 ELSE 0 END as is_key_account_flag,
    CASE WHEN w.reconciliation_status = 'High Variance' THEN 1 WHEN w.variance_amount > COALESCE(w.period_p95_variance, 0) * 1.5 THEN 1 ELSE 0 END as is_anomaly_label
FROM windowed_features w
"""

session.sql(feature_store_sql).collect()
print("Feature Store View Created: COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES")

In [ ]:
feature_stats = session.sql("""
SELECT COUNT(*) as feature_count, 
       COUNT(DISTINCT assignment_id) as unique_assignments,
       AVG(variance_z_score) as avg_z_score,
       SUM(is_anomaly_label) as labeled_anomalies,
       ROUND(100.0 * SUM(is_anomaly_label) / COUNT(*), 2) as anomaly_rate_pct
FROM COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES
""").to_pandas()

print("Feature Store Statistics:")
print(feature_stats.to_string(index=False))

---
## 3. Load Training Data with Temporal Split

**Critical**: To prevent data leakage, we use temporal splitting:
- **Training set**: Earlier periods (first 80% of time)
- **Validation set**: Later periods (last 20% of time)

This simulates real-world deployment where we train on historical data and predict future anomalies.

In [ ]:
date_range = session.sql("""
SELECT 
    MIN(period_end_date) as min_date,
    MAX(period_end_date) as max_date,
    COUNT(DISTINCT period_end_date) as num_periods
FROM COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES
""").to_pandas()

print(f"Date Range: {date_range['MIN_DATE'].iloc[0]} to {date_range['MAX_DATE'].iloc[0]}")
print(f"Number of periods: {date_range['NUM_PERIODS'].iloc[0]}")

In [ ]:
periods_df = session.sql("""
SELECT DISTINCT period_end_date 
FROM COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES 
ORDER BY period_end_date
""").to_pandas()

num_periods = len(periods_df)
split_idx = int(num_periods * 0.8)
split_date = periods_df.iloc[split_idx]['PERIOD_END_DATE']

print(f"Total periods: {num_periods}")
print(f"Training periods: {split_idx} (before {split_date})")
print(f"Validation periods: {num_periods - split_idx} (on/after {split_date})")

In [ ]:
FEATURE_COLUMNS = [
    'variance_amount', 'gl_bank_diff', 'gl_subledger_diff', 'recon_count', 'unidentified_amount',
    'variance_pct_change', 'variance_z_score', 'variance_z_score_period', 'variance_pct_of_balance',
    'variance_vs_rolling_max', 'is_above_p95', 'gl_bank_diff_ratio', 'balance_change_pct',
    'hierarchy_depth_normalized', 'is_key_account_flag', 'rolling_avg_variance_3', 'rolling_std_variance_3',
    'rolling_max_variance_6', 'entity_avg_variance', 'entity_assignment_count'
]

feature_cols_str = ', '.join(FEATURE_COLUMNS)

train_df = session.sql(f"""
SELECT assignment_id, period_id, period_end_date, entity_name, account_combination,
       {feature_cols_str}, is_anomaly_label
FROM COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES
WHERE period_end_date < '{split_date}'
""").to_pandas()

val_df = session.sql(f"""
SELECT assignment_id, period_id, period_end_date, entity_name, account_combination,
       {feature_cols_str}, is_anomaly_label
FROM COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES
WHERE period_end_date >= '{split_date}'
""").to_pandas()

print(f"Training set: {len(train_df):,} records")
print(f"Validation set: {len(val_df):,} records")
print(f"\nTraining anomaly rate: {train_df['IS_ANOMALY_LABEL'].mean()*100:.2f}%")
print(f"Validation anomaly rate: {val_df['IS_ANOMALY_LABEL'].mean()*100:.2f}%")

In [ ]:
X_train = train_df[FEATURE_COLUMNS].fillna(0).replace([np.inf, -np.inf], 0)
y_train = train_df['IS_ANOMALY_LABEL']

X_val = val_df[FEATURE_COLUMNS].fillna(0).replace([np.inf, -np.inf], 0)
y_val = val_df['IS_ANOMALY_LABEL']

print(f"Training features shape: {X_train.shape}")
print(f"Validation features shape: {X_val.shape}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("\nFeatures scaled successfully")

---
## 4. Experiment Tracking Setup

Using Snowflake's native experiment tracking to log metrics, parameters, and artifacts.

In [ ]:
from snowflake.ml.experiment import Experiment
from snowflake.ml.registry import Registry
import json

EXPERIMENT_NAME = "reconciliation_anomaly_detection"
MODEL_NAME = "anomaly_detector"

experiment = Experiment(
    session=session,
    name=EXPERIMENT_NAME,
    database="COCO_LIVE_DB",
    schema="DBT"
)

print(f"Experiment created: {EXPERIMENT_NAME}")
print(f"Database: COCO_LIVE_DB.DBT")

---
## 5. Model Training with Hyperparameter Tuning

We'll train multiple anomaly detection models:
1. **Isolation Forest** - Tree-based anomaly detection
2. **One-Class SVM** - Kernel-based novelty detection  
3. **Local Outlier Factor** - Density-based detection

Using cross-validation and grid search for hyperparameter optimization.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import time

def evaluate_model(y_true, y_pred, y_scores=None):
    """Calculate evaluation metrics for anomaly detection."""
    y_pred_binary = (y_pred == -1).astype(int)
    
    metrics = {
        'precision': precision_score(y_true, y_pred_binary, zero_division=0),
        'recall': recall_score(y_true, y_pred_binary, zero_division=0),
        'f1_score': f1_score(y_true, y_pred_binary, zero_division=0),
    }
    
    if y_scores is not None:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, -y_scores)
        except:
            metrics['roc_auc'] = 0.0
    
    cm = confusion_matrix(y_true, y_pred_binary)
    metrics['true_negatives'] = cm[0, 0] if cm.shape[0] > 1 else 0
    metrics['false_positives'] = cm[0, 1] if cm.shape[1] > 1 else 0
    metrics['false_negatives'] = cm[1, 0] if cm.shape[0] > 1 else 0
    metrics['true_positives'] = cm[1, 1] if cm.shape == (2, 2) else 0
    
    return metrics

print("Evaluation function defined")

In [ ]:
isolation_forest_params = {
    'n_estimators': [100, 200, 300],
    'max_samples': ['auto', 0.5, 0.8],
    'contamination': [0.05, 0.10, 0.15],
    'max_features': [0.5, 0.8, 1.0],
    'random_state': [42]
}

print("Isolation Forest Parameter Grid:")
for param, values in isolation_forest_params.items():
    print(f"  {param}: {values}")
print(f"\nTotal combinations: {len(list(ParameterGrid(isolation_forest_params)))}")

In [ ]:
print("="*60)
print("TRAINING ISOLATION FOREST MODELS")
print("="*60)

if_results = []
best_if_f1 = 0
best_if_model = None
best_if_params = None

tscv = TimeSeriesSplit(n_splits=3)

for i, params in enumerate(ParameterGrid(isolation_forest_params)):
    if i >= 15:
        break
    
    start_time = time.time()
    
    cv_scores = []
    for train_idx, test_idx in tscv.split(X_train_scaled):
        X_cv_train, X_cv_test = X_train_scaled[train_idx], X_train_scaled[test_idx]
        y_cv_train, y_cv_test = y_train.iloc[train_idx], y_train.iloc[test_idx]
        
        model = IsolationForest(**params)
        model.fit(X_cv_train)
        y_pred = model.predict(X_cv_test)
        y_scores = model.decision_function(X_cv_test)
        
        metrics = evaluate_model(y_cv_test, y_pred, y_scores)
        cv_scores.append(metrics['f1_score'])
    
    avg_cv_f1 = np.mean(cv_scores)
    
    final_model = IsolationForest(**params)
    final_model.fit(X_train_scaled)
    y_val_pred = final_model.predict(X_val_scaled)
    y_val_scores = final_model.decision_function(X_val_scaled)
    val_metrics = evaluate_model(y_val, y_val_pred, y_val_scores)
    
    train_time = time.time() - start_time
    
    result = {
        'model_type': 'IsolationForest',
        'params': params,
        'cv_f1_mean': avg_cv_f1,
        'cv_f1_std': np.std(cv_scores),
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': train_time
    }
    if_results.append(result)
    
    if val_metrics['f1_score'] > best_if_f1:
        best_if_f1 = val_metrics['f1_score']
        best_if_model = final_model
        best_if_params = params
    
    if (i + 1) % 5 == 0:
        print(f"Completed {i+1} configurations, best F1: {best_if_f1:.4f}")

print(f"\nBest Isolation Forest F1: {best_if_f1:.4f}")
print(f"Best params: {best_if_params}")

In [ ]:
ocsvm_params = {
    'kernel': ['rbf', 'poly'],
    'nu': [0.05, 0.10, 0.15],
    'gamma': ['scale', 'auto', 0.1]
}

print("="*60)
print("TRAINING ONE-CLASS SVM MODELS")
print("="*60)

ocsvm_results = []
best_ocsvm_f1 = 0
best_ocsvm_model = None
best_ocsvm_params = None

X_train_subset = X_train_scaled[:50000] if len(X_train_scaled) > 50000 else X_train_scaled
y_train_subset = y_train.iloc[:50000] if len(y_train) > 50000 else y_train
X_val_subset = X_val_scaled[:20000] if len(X_val_scaled) > 20000 else X_val_scaled
y_val_subset = y_val.iloc[:20000] if len(y_val) > 20000 else y_val

for i, params in enumerate(ParameterGrid(ocsvm_params)):
    if i >= 9:
        break
    
    start_time = time.time()
    
    model = OneClassSVM(**params)
    model.fit(X_train_subset)
    y_val_pred = model.predict(X_val_subset)
    y_val_scores = model.decision_function(X_val_subset)
    val_metrics = evaluate_model(y_val_subset, y_val_pred, y_val_scores)
    
    train_time = time.time() - start_time
    
    result = {
        'model_type': 'OneClassSVM',
        'params': params,
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': train_time
    }
    ocsvm_results.append(result)
    
    if val_metrics['f1_score'] > best_ocsvm_f1:
        best_ocsvm_f1 = val_metrics['f1_score']
        best_ocsvm_model = model
        best_ocsvm_params = params
    
    print(f"Config {i+1}: F1={val_metrics['f1_score']:.4f}, Time={train_time:.1f}s")

print(f"\nBest One-Class SVM F1: {best_ocsvm_f1:.4f}")
print(f"Best params: {best_ocsvm_params}")

In [ ]:
lof_params = {
    'n_neighbors': [10, 20, 50],
    'contamination': [0.05, 0.10, 0.15],
    'metric': ['euclidean', 'manhattan']
}

print("="*60)
print("TRAINING LOCAL OUTLIER FACTOR MODELS")
print("="*60)

lof_results = []
best_lof_f1 = 0
best_lof_model = None
best_lof_params = None

for i, params in enumerate(ParameterGrid(lof_params)):
    if i >= 9:
        break
    
    start_time = time.time()
    
    model = LocalOutlierFactor(**params, novelty=True)
    model.fit(X_train_subset)
    y_val_pred = model.predict(X_val_subset)
    y_val_scores = model.decision_function(X_val_subset)
    val_metrics = evaluate_model(y_val_subset, y_val_pred, y_val_scores)
    
    train_time = time.time() - start_time
    
    result = {
        'model_type': 'LocalOutlierFactor',
        'params': params,
        'val_precision': val_metrics['precision'],
        'val_recall': val_metrics['recall'],
        'val_f1': val_metrics['f1_score'],
        'val_roc_auc': val_metrics.get('roc_auc', 0),
        'train_time': train_time
    }
    lof_results.append(result)
    
    if val_metrics['f1_score'] > best_lof_f1:
        best_lof_f1 = val_metrics['f1_score']
        best_lof_model = model
        best_lof_params = params
    
    print(f"Config {i+1}: F1={val_metrics['f1_score']:.4f}, Time={train_time:.1f}s")

print(f"\nBest LOF F1: {best_lof_f1:.4f}")
print(f"Best params: {best_lof_params}")

---
## 6. Results Comparison & Visualization

In [ ]:
all_results = if_results + ocsvm_results + lof_results
results_df = pd.DataFrame(all_results)

summary = results_df.groupby('model_type').agg({
    'val_f1': ['max', 'mean', 'std'],
    'val_precision': 'max',
    'val_recall': 'max',
    'val_roc_auc': 'max',
    'train_time': 'mean'
}).round(4)

print("Model Comparison Summary:")
print("="*70)
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
for model_type in results_df['model_type'].unique():
    model_data = results_df[results_df['model_type'] == model_type]
    ax1.scatter(model_data['val_precision'], model_data['val_recall'], 
                label=model_type, s=100, alpha=0.7)
ax1.set_xlabel('Precision')
ax1.set_ylabel('Recall')
ax1.set_title('Precision vs Recall by Model Type')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
best_f1_by_model = results_df.loc[results_df.groupby('model_type')['val_f1'].idxmax()]
colors = ['steelblue', 'coral', 'green']
ax2.bar(best_f1_by_model['model_type'], best_f1_by_model['val_f1'], color=colors)
ax2.set_xlabel('Model Type')
ax2.set_ylabel('Best F1 Score')
ax2.set_title('Best F1 Score by Model Type')
ax2.tick_params(axis='x', rotation=45)
for i, (idx, row) in enumerate(best_f1_by_model.iterrows()):
    ax2.text(i, row['val_f1'] + 0.01, f"{row['val_f1']:.3f}", ha='center')

ax3 = axes[1, 0]
ax3.boxplot([results_df[results_df['model_type'] == m]['val_f1'] for m in results_df['model_type'].unique()],
            labels=results_df['model_type'].unique())
ax3.set_ylabel('F1 Score')
ax3.set_title('F1 Score Distribution by Model')
ax3.tick_params(axis='x', rotation=45)

ax4 = axes[1, 1]
metrics = ['val_precision', 'val_recall', 'val_f1', 'val_roc_auc']
x = np.arange(len(metrics))
width = 0.25
for i, model_type in enumerate(best_f1_by_model['model_type'].unique()):
    model_row = best_f1_by_model[best_f1_by_model['model_type'] == model_type].iloc[0]
    values = [model_row[m] for m in metrics]
    ax4.bar(x + i*width, values, width, label=model_type)
ax4.set_xlabel('Metric')
ax4.set_ylabel('Score')
ax4.set_title('Best Model Metrics Comparison')
ax4.set_xticks(x + width)
ax4.set_xticklabels(['Precision', 'Recall', 'F1', 'ROC-AUC'])
ax4.legend()

plt.tight_layout()
plt.savefig('/tmp/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMetrics visualization saved to /tmp/model_comparison.png")

---
## 7. Log Experiments & Register Models

In [ ]:
best_models = {
    'IsolationForest': (best_if_model, best_if_params, best_if_f1),
    'OneClassSVM': (best_ocsvm_model, best_ocsvm_params, best_ocsvm_f1),
    'LocalOutlierFactor': (best_lof_model, best_lof_params, best_lof_f1)
}

for model_name, (model, params, f1) in best_models.items():
    if model is None:
        continue
    
    run = experiment.start_run(run_name=f"{model_name}_best")
    
    run.log_param("model_type", model_name)
    for param_name, param_value in params.items():
        run.log_param(param_name, str(param_value))
    
    y_val_pred_final = model.predict(X_val_scaled if model_name == 'IsolationForest' else X_val_subset)
    y_val_scores_final = model.decision_function(X_val_scaled if model_name == 'IsolationForest' else X_val_subset)
    y_val_final = y_val if model_name == 'IsolationForest' else y_val_subset
    
    final_metrics = evaluate_model(y_val_final, y_val_pred_final, y_val_scores_final)
    
    run.log_metric("precision", final_metrics['precision'])
    run.log_metric("recall", final_metrics['recall'])
    run.log_metric("f1_score", final_metrics['f1_score'])
    run.log_metric("roc_auc", final_metrics.get('roc_auc', 0))
    run.log_metric("true_positives", final_metrics['true_positives'])
    run.log_metric("false_positives", final_metrics['false_positives'])
    
    run.end_run()
    print(f"Logged {model_name}: F1={final_metrics['f1_score']:.4f}")

print("\nAll experiments logged successfully!")

In [ ]:
overall_best_f1 = 0
overall_best_model = None
overall_best_name = None
overall_best_params = None

for model_name, (model, params, f1) in best_models.items():
    if model is not None and f1 > overall_best_f1:
        overall_best_f1 = f1
        overall_best_model = model
        overall_best_name = model_name
        overall_best_params = params

print(f"Overall Best Model: {overall_best_name}")
print(f"F1 Score: {overall_best_f1:.4f}")
print(f"Parameters: {overall_best_params}")

In [ ]:
registry = Registry(session=session, database_name="COCO_LIVE_DB", schema_name="DBT")

print(f"Model Registry initialized: COCO_LIVE_DB.DBT")

In [ ]:
import pickle
from snowflake.ml.model import ModelVersion

model_versions = {}

for model_name, (model, params, f1) in best_models.items():
    if model is None:
        continue
    
    registry_name = f"ANOMALY_DETECTOR_{model_name.upper()}"
    
    sample_input = pd.DataFrame(X_train_scaled[:5], columns=FEATURE_COLUMNS)
    
    y_pred_sample = model.predict(X_train_scaled[:5])
    y_scores_sample = model.decision_function(X_train_scaled[:5])
    
    model_info = registry.log_model(
        model=model,
        model_name=registry_name,
        version_name="v1",
        sample_input_data=sample_input,
        metrics={
            "f1_score": float(f1),
            "model_type": model_name
        },
        comment=f"{model_name} anomaly detector for reconciliation variances. Params: {params}"
    )
    
    model_versions[model_name] = model_info
    print(f"Registered: {registry_name} v1 (F1={f1:.4f})")

print("\nAll models registered to Snowflake Model Registry!")

---
## 8. Model Promotion - Set Best as Default

In [ ]:
print("="*60)
print("MODEL PROMOTION")
print("="*60)

all_registered_models = []

for model_name, mv in model_versions.items():
    _, (_, _, f1) = [(k, v) for k, v in best_models.items() if k == model_name][0]
    all_registered_models.append({
        'name': model_name,
        'registry_name': f"ANOMALY_DETECTOR_{model_name.upper()}",
        'f1_score': f1,
        'model_version': mv
    })

all_registered_models.sort(key=lambda x: x['f1_score'], reverse=True)

print("\nRegistered Models (sorted by F1):")
for i, m in enumerate(all_registered_models):
    marker = "👑 BEST" if i == 0 else ""
    print(f"  {i+1}. {m['registry_name']}: F1={m['f1_score']:.4f} {marker}")

best_registered = all_registered_models[0]
print(f"\nPromoting {best_registered['registry_name']} as DEFAULT model")

In [ ]:
best_model_ref = registry.get_model(best_registered['registry_name'])
best_model_ref.default = "v1"

print(f"\n✓ {best_registered['registry_name']} v1 set as DEFAULT")
print(f"\nTo use this model for inference:")
print(f"  mv = registry.get_model('{best_registered['registry_name']}')")
print(f"  predictions = mv.run(input_df, function_name='predict')")

---
## 9. Generate Final Metrics Report

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

ax1 = axes[0, 0]
status_dist.plot(kind='pie', y='COUNT', labels=status_dist['RECONCILIATION_STATUS'], ax=ax1, autopct='%1.1f%%')
ax1.set_title('Reconciliation Status Distribution')
ax1.set_ylabel('')

ax2 = axes[0, 1]
for model_type in results_df['model_type'].unique():
    model_data = results_df[results_df['model_type'] == model_type]
    ax2.scatter(model_data['val_precision'], model_data['val_recall'], label=model_type, s=80, alpha=0.7)
ax2.set_xlabel('Precision')
ax2.set_ylabel('Recall')
ax2.set_title('Model Performance: Precision vs Recall')
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[0, 2]
model_names = [m['registry_name'].replace('ANOMALY_DETECTOR_', '') for m in all_registered_models]
f1_scores = [m['f1_score'] for m in all_registered_models]
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(model_names))]
ax3.barh(model_names, f1_scores, color=colors)
ax3.set_xlabel('F1 Score')
ax3.set_title('Final Model Comparison')
for i, (name, f1) in enumerate(zip(model_names, f1_scores)):
    ax3.text(f1 + 0.01, i, f'{f1:.3f}', va='center')

ax4 = axes[1, 0]
if best_if_model is not None:
    y_scores_final = best_if_model.decision_function(X_val_scaled)
    ax4.hist(y_scores_final[y_val == 0], bins=50, alpha=0.7, label='Normal', density=True)
    ax4.hist(y_scores_final[y_val == 1], bins=50, alpha=0.7, label='Anomaly', density=True)
    ax4.set_xlabel('Anomaly Score')
    ax4.set_ylabel('Density')
    ax4.set_title('Score Distribution (Best Model)')
    ax4.legend()

ax5 = axes[1, 1]
feature_importance = np.abs(X_train_scaled).mean(axis=0)
top_features_idx = np.argsort(feature_importance)[-10:]
top_features = [FEATURE_COLUMNS[i] for i in top_features_idx]
top_importance = feature_importance[top_features_idx]
ax5.barh(top_features, top_importance, color='steelblue')
ax5.set_xlabel('Mean Absolute Value')
ax5.set_title('Top 10 Features (by mean abs value)')

ax6 = axes[1, 2]
ax6.axis('off')
summary_text = f"""
ANOMALY DETECTION SUMMARY
========================

Best Model: {best_registered['registry_name']}
F1 Score: {best_registered['f1_score']:.4f}

Training Records: {len(train_df):,}
Validation Records: {len(val_df):,}
Features Used: {len(FEATURE_COLUMNS)}

Models Evaluated:
- Isolation Forest: {best_if_f1:.4f}
- One-Class SVM: {best_ocsvm_f1:.4f}
- Local Outlier Factor: {best_lof_f1:.4f}

Experiment: {EXPERIMENT_NAME}
Registry: COCO_LIVE_DB.DBT
"""
ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.savefig('/tmp/anomaly_detection_report.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal report saved to /tmp/anomaly_detection_report.png")

In [ ]:
print("\n" + "="*70)
print("ANOMALY DETECTION PIPELINE COMPLETE")
print("="*70)

print(f"""
✅ EDA Complete
   - Analyzed {len(train_df) + len(val_df):,} records across {date_range['NUM_PERIODS'].iloc[0]} periods
   
✅ Feature Store Created
   - View: COCO_LIVE_DB.DBT.ANOMALY_DETECTION_FEATURES
   - {len(FEATURE_COLUMNS)} engineered features
   
✅ Model Training Complete
   - Isolation Forest: {len(if_results)} configurations tested
   - One-Class SVM: {len(ocsvm_results)} configurations tested
   - Local Outlier Factor: {len(lof_results)} configurations tested
   
✅ Experiment Tracking
   - Experiment: {EXPERIMENT_NAME}
   - Location: COCO_LIVE_DB.DBT
   
✅ Model Registry
   - ANOMALY_DETECTOR_ISOLATIONFOREST v1
   - ANOMALY_DETECTOR_ONECLASSSVM v1
   - ANOMALY_DETECTOR_LOCALOUTLIERFACTOR v1
   
✅ Model Promotion
   - Default Model: {best_registered['registry_name']}
   - F1 Score: {best_registered['f1_score']:.4f}
   
📊 Artifacts
   - /tmp/model_comparison.png
   - /tmp/anomaly_detection_report.png
""")

print("\nNext Steps:")
print("1. Deploy model for real-time scoring")
print("2. Set up model monitoring for drift detection")
print("3. Create Streamlit dashboard for anomaly alerts")